# 01 – Load and combine CFPB bank complaint data
Source: CFPB Consumer Complaint Database, downloaded in slices (website export limit is 100,000 rows).
Scope: Checking/savings accounts and credit cards, complaints received 1 Jan 2023 – Nov 2026.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

files = sorted(RAW.glob("*.csv"))
for f in files:
    print(f.name)

checking_2023.csv
checking_2024.csv
checking_2025.csv
checking_2026.csv
credit_card_2023.csv
credit_card_2023_old_label.csv
credit_card_2024.csv
credit_card_2025.csv
credit_card_2026.csv


In [2]:
frames = []
for f in files:
    df_part = pd.read_csv(f, low_memory=False)
    df_part["source_file"] = f.name
    frames.append(df_part)
    print(f"{f.name}: {len(df_part):,} rows")

df = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {len(df):,} rows")

checking_2023.csv: 49,873 rows
checking_2024.csv: 52,821 rows
checking_2025.csv: 84,195 rows
checking_2026.csv: 78,582 rows
credit_card_2023.csv: 21,872 rows
credit_card_2023_old_label.csv: 33,905 rows
credit_card_2024.csv: 75,989 rows
credit_card_2025.csv: 89,978 rows
credit_card_2026.csv: 84,576 rows

Total: 571,791 rows


In [3]:
credit_subproducts = ["General-purpose credit card or charge card", "Store credit card"]

old_label = df["Product"] == "Credit card or prepaid card"
prepaid = old_label & ~df["Sub-product"].isin(credit_subproducts)

df = df[~prepaid].copy()
df.loc[df["Product"] == "Credit card or prepaid card", "Product"] = "Credit card"

print(f"Dropped prepaid rows: {prepaid.sum():,}")
print(f"Rows now: {len(df):,}")
print(df["Product"].value_counts())

Dropped prepaid rows: 2,653
Rows now: 569,138
Product
Credit card                    303667
Checking or savings account    265471
Name: count, dtype: int64


In [4]:
assert df["Complaint ID"].is_unique, "Duplicate complaint IDs found!"
df.to_csv(PROCESSED / "complaints_combined.csv", index=False)
print("Saved:", PROCESSED / "complaints_combined.csv")

Saved: ..\data\processed\complaints_combined.csv
